# LeetCode 997 — Find the Town Judge bằng Python

**Link đề:** https://leetcode.com/problems/find-the-town-judge/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Graph Theory

```python
class Solution:
    def findJudge(self, n: int, trust: list[list[int]]) -> int:
        ...
```

## Mục tiêu học
1. Nhìn bài toán dưới dạng **đồ thị có hướng** (directed graph): mỗi người là một đỉnh, mỗi `[a, b]` là một **cạnh** `a → b`.
2. Hiểu **bậc vào** (in-degree) và **bậc ra** (out-degree), và biểu diễn chúng bằng mảng đếm.
3. Rút gọn hai mảng thành **một điểm số** `in − out` và hiểu vì sao chỉ thẩm phán đạt `n − 1`.
4. Viết 4 cách: brute force kiểm từng người, hai mảng bậc, một mảng điểm số, dùng set + loại trừ.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Thị trấn có `n` người, đánh số `1..n`. Có tin đồn một người là **thẩm phán** bí mật. Nếu thẩm phán tồn tại thì:
1. Thẩm phán **không tin ai**.
2. **Mọi người khác** đều tin thẩm phán.
3. Có **đúng một** người thoả cả 1 và 2.

`trust[i] = [a, b]` nghĩa là người `a` tin người `b`. Trả về nhãn của thẩm phán nếu tồn tại và xác định được, ngược lại `-1`.

| n | trust | Kết quả | Giải thích |
|:---:|---|:---:|---|
| 2 | `[[1,2]]` | `2` | 1 tin 2, 2 không tin ai |
| 3 | `[[1,3],[2,3]]` | `3` | 1, 2 đều tin 3; 3 không tin ai |
| 3 | `[[1,3],[2,3],[3,1]]` | `-1` | 3 tin 1 → vi phạm điều kiện 1 |

**Constraints:** `1 <= n <= 1000`, `0 <= len(trust) <= 10^4`, các cặp không trùng, `a != b`, `1 <= a, b <= n`.

Chú ý trường hợp `n = 1`, `trust = []`: người duy nhất không tin ai, và "mọi người khác" là tập rỗng → **người 1 là thẩm phán**.

## 2. Nền tảng: đồ thị, bậc vào, bậc ra

Vẽ mỗi người là một chấm, mỗi `[a, b]` là mũi tên `a → b`.

- **Bậc ra** `out[p]` = số mũi tên **đi ra** từ `p` = số người `p` tin.
- **Bậc vào** `in[p]` = số mũi tên **đi vào** `p` = số người tin `p`.

Thẩm phán ⇔ `out = 0` (không tin ai) **và** `in = n − 1` (mọi người khác đều tin).

In [1]:
n = 3
trust = [[1, 3], [2, 3]]
for a, b_ in trust:
    print(f"{a} -> {b_}  (người {a} tin người {b_})")

1 -> 3  (người 1 tin người 3)
2 -> 3  (người 2 tin người 3)


### 2.1. Mảng đếm theo nhãn `1..n`

Nhãn bắt đầu từ 1, nên ta tạo mảng dài `n + 1` và **bỏ trống ô 0** — khi đó `count[p]` ứng đúng với người `p`, không phải trừ 1.

`for a, b in trust:` — mỗi phần tử của `trust` là list 2 phần tử, được **unpack** vào `a`, `b`.

In [2]:
n = 3
trust = [[1, 3], [2, 3]]
in_deg = [0] * (n + 1)
out_deg = [0] * (n + 1)
for a, b_ in trust:
    out_deg[a] += 1
    in_deg[b_] += 1
print("in_deg  (ô 0 bỏ trống):", in_deg)
print("out_deg (ô 0 bỏ trống):", out_deg)
for p in range(1, n + 1):
    print(f"người {p}: tin {out_deg[p]} người, được {in_deg[p]} người tin")

in_deg  (ô 0 bỏ trống): [0, 0, 0, 2]
out_deg (ô 0 bỏ trống): [0, 1, 1, 0]
người 1: tin 1 người, được 0 người tin
người 2: tin 1 người, được 0 người tin
người 3: tin 0 người, được 2 người tin


### 2.2. `range(1, n + 1)`

`range(1, n + 1)` sinh `1, 2, ..., n` (không gồm `n + 1`) — đúng tập nhãn người.

In [3]:
print(list(range(1, 4)))

[1, 2, 3]


## 3. Cách trực giác (baseline) — thử từng người làm thẩm phán

Với mỗi ứng viên `c`:
- Duyệt `trust`: nếu có `[c, *]` → `c` tin ai đó → loại.
- Đếm số `[*, c]`; phải bằng `n − 1`.

Nếu có đúng một ứng viên qua → trả về; không thì `-1`.

In [4]:
def find_judge_brute(n: int, trust: list[list[int]]) -> int:
    judges = []
    for c in range(1, n + 1):
        trusts_someone = any(a == c for a, b in trust)
        trusted_by = sum(1 for a, b in trust if b == c)
        if not trusts_someone and trusted_by == n - 1:
            judges.append(c)
    return judges[0] if len(judges) == 1 else -1

print(find_judge_brute(2, [[1, 2]]))
print(find_judge_brute(3, [[1, 3], [2, 3]]))
print(find_judge_brute(3, [[1, 3], [2, 3], [3, 1]]))
print(find_judge_brute(1, []))

2
3
-1
1


- `any(...)` trả `True` nếu **ít nhất một** phần tử là `True` (dừng sớm).
- `sum(1 for ... if ...)` đếm số phần tử thoả điều kiện.

Độ phức tạp: `n` ứng viên × duyệt `trust` (`m` cặp) → **O(n·m)** = 10³ × 10⁴ = 10⁷. Chạy được nhưng thừa.

In [5]:
# Thí nghiệm nhỏ với any và sum(1 for ...)
trust = [[1, 3], [2, 3], [3, 1]]
c = 3
print("3 có tin ai không?", any(a == c for a, b in trust))
print("Số người tin 3   :", sum(1 for a, b in trust if b == c))
print("any([]) =", any([]), "| sum(()) =", sum(()))   # dãy rỗng: any -> False, sum -> 0

3 có tin ai không? True
Số người tin 3   : 2
any([]) = False | sum(()) = 0


Chú ý dòng cuối: với `trust = []`, `any(...)` là `False` và tổng là `0` — nên brute force tự xử lý đúng trường hợp `n = 1` (cần `0` người tin, và người 1 không tin ai).

## 4. Nút thắt

Mỗi ứng viên lại duyệt **toàn bộ** `trust`. Nhưng thông tin cần cho mọi ứng viên (bậc vào, bậc ra) có thể **tính chung trong một lượt** duyệt `trust` → O(n + m).

## 5. Insight — gộp thành một điểm số

Định nghĩa `score[p] = in[p] − out[p]`.

- Thẩm phán: `in = n − 1`, `out = 0` → `score = n − 1`.
- Với **bất kỳ** ai: `in[p] ≤ n − 1` (chỉ có `n − 1` người khác, không ai tự tin mình, cặp không trùng) và `out[p] ≥ 0`. Nên `score[p] ≤ n − 1`, và **dấu bằng chỉ xảy ra** khi `in[p] = n − 1` **và** `out[p] = 0`.

➡ `score[p] == n − 1` ⇔ `p` thoả điều kiện 1 và 2. Một mảng là đủ!

Còn điều kiện 3 (duy nhất)? Nếu hai người `p ≠ q` cùng có `score = n − 1` thì `q` tin `p` (vì `in[p] = n − 1`) → `out[q] ≥ 1` → mâu thuẫn. Vậy **tối đa một** người đạt `n − 1`.

In [6]:
n = 3
for trust in ([[1, 3], [2, 3]], [[1, 3], [2, 3], [3, 1]]):
    score = [0] * (n + 1)
    for a, b_ in trust:
        score[a] -= 1
        score[b_] += 1
    print(f"trust={trust} -> score[1..n] = {score[1:]}, cần đạt n-1 = {n - 1}")

trust=[[1, 3], [2, 3]] -> score[1..n] = [-1, -1, 2], cần đạt n-1 = 2
trust=[[1, 3], [2, 3], [3, 1]] -> score[1..n] = [0, -1, 1], cần đạt n-1 = 2


## 6. Lời giải chính — xây từng đoạn rồi ghép

**Đoạn 1 — lọc nhanh (tuỳ chọn):** thẩm phán cần `n − 1` người tin → phải có ít nhất `n − 1` cặp. Nếu `len(trust) < n − 1` → `-1` ngay.

**Đoạn 2 — tính điểm:** `score[a] -= 1` (a tin ai đó), `score[b] += 1` (b được tin).

**Đoạn 3 — tìm người đạt `n − 1`.**

In [7]:
def find_judge(n: int, trust: list[list[int]]) -> int:
    if len(trust) < n - 1:                 # Đoạn 1
        return -1
    score = [0] * (n + 1)                  # Đoạn 2
    for a, b in trust:
        score[a] -= 1
        score[b] += 1
    for p in range(1, n + 1):              # Đoạn 3
        if score[p] == n - 1:
            return p
    return -1

print(find_judge(2, [[1, 2]]))
print(find_judge(3, [[1, 3], [2, 3]]))
print(find_judge(3, [[1, 3], [2, 3], [3, 1]]))
print(find_judge(1, []))

2
3
-1
1


## 7. Trace từng bước

`n = 4`, `trust = [[1,3],[1,4],[2,3],[2,4],[4,3]]` → mong đợi `3`.

| Cặp | score[1] | score[2] | score[3] | score[4] |
|---|:---:|:---:|:---:|:---:|
| bắt đầu | 0 | 0 | 0 | 0 |
| [1,3] | −1 | 0 | 1 | 0 |
| [1,4] | −2 | 0 | 1 | 1 |
| [2,3] | −2 | −1 | 2 | 1 |
| [2,4] | −2 | −2 | 2 | 2 |
| [4,3] | −2 | −2 | **3** | 1 |

Chỉ người 3 đạt `n − 1 = 3` → thẩm phán là **3**.

In [8]:
def trace_judge(n: int, trust: list[list[int]]) -> int:
    print(f"n={n}, cần score = {n - 1}")
    if len(trust) < n - 1:
        print("  quá ít cặp tin tưởng -> -1")
        return -1
    score = [0] * (n + 1)
    for a, b in trust:
        score[a] -= 1
        score[b] += 1
        print(f"  [{a},{b}]: score[1..n] = {score[1:]}")
    for p in range(1, n + 1):
        if score[p] == n - 1:
            print(f"  người {p} đạt {n - 1} -> thẩm phán")
            return p
    print("  không ai đạt -> -1")
    return -1

trace_judge(4, [[1, 3], [1, 4], [2, 3], [2, 4], [4, 3]])
print()
trace_judge(3, [[1, 3], [2, 3], [3, 1]])
print()
trace_judge(3, [[1, 2]])

n=4, cần score = 3
  [1,3]: score[1..n] = [-1, 0, 1, 0]
  [1,4]: score[1..n] = [-2, 0, 1, 1]
  [2,3]: score[1..n] = [-2, -1, 2, 1]
  [2,4]: score[1..n] = [-2, -2, 2, 2]
  [4,3]: score[1..n] = [-2, -2, 3, 1]
  người 3 đạt 3 -> thẩm phán

n=3, cần score = 2
  [1,3]: score[1..n] = [-1, 0, 1]
  [2,3]: score[1..n] = [-1, -1, 2]
  [3,1]: score[1..n] = [0, -1, 1]
  không ai đạt -> -1

n=3, cần score = 2
  quá ít cặp tin tưởng -> -1


-1

Ví dụ cuối dừng ở **Đoạn 1**: chỉ 1 cặp nhưng cần ít nhất 2.

## 8. Code nộp LeetCode

### Cách A — một mảng điểm số

In [9]:
class Solution:
    def findJudge(self, n: int, trust: list[list[int]]) -> int:
        if len(trust) < n - 1:
            return -1
        score = [0] * (n + 1)
        for a, b in trust:
            score[a] -= 1
            score[b] += 1
        for p in range(1, n + 1):
            if score[p] == n - 1:
                return p
        return -1

s = Solution()
print(s.findJudge(2, [[1, 2]]), s.findJudge(3, [[1, 3], [2, 3]]), s.findJudge(3, [[1, 3], [2, 3], [3, 1]]))

2 3 -1


### Cách B — hai mảng bậc vào / bậc ra (rõ nghĩa nhất)

In [10]:
class SolutionDegrees:
    def findJudge(self, n: int, trust: list[list[int]]) -> int:
        in_deg = [0] * (n + 1)
        out_deg = [0] * (n + 1)
        for a, b in trust:
            out_deg[a] += 1
            in_deg[b] += 1
        for p in range(1, n + 1):
            if in_deg[p] == n - 1 and out_deg[p] == 0:
                return p
        return -1

print(SolutionDegrees().findJudge(3, [[1, 3], [2, 3]]))

3


### Cách C — set "những người tin ai đó" + Counter

- Ứng viên phải **không** nằm trong tập `{a for a, b in trust}`.
- Và phải được `n − 1` người tin: dùng `Counter(b for a, b in trust)`.

`{a for a, b in trust}` là **set comprehension** — giống list comprehension nhưng dùng `{}` và tự loại trùng.

In [11]:
from collections import Counter

class SolutionSet:
    def findJudge(self, n: int, trust: list[list[int]]) -> int:
        trusters = {a for a, b in trust}
        trusted = Counter(b for a, b in trust)
        for p in range(1, n + 1):
            if p not in trusters and trusted[p] == n - 1:
                return p
        return -1

print({a for a, b in [[1, 3], [2, 3], [1, 2]]})
print(SolutionSet().findJudge(3, [[1, 3], [2, 3]]))

{1, 2}
3


`Counter` trả `0` cho khóa chưa có (`trusted[p]` không gây `KeyError`) — khác với dict thường.

### Cách D — loại trừ ứng viên (giống "celebrity problem")

Ứng viên duy nhất có thể là thẩm phán là người **không nằm trong** tập `trusters`. Nếu có nhiều hơn một người như vậy → không thể ai được mọi người khác tin (vì những người không tin ai thì không tin nhau) → `-1`.

In [12]:
class SolutionEliminate:
    def findJudge(self, n: int, trust: list[list[int]]) -> int:
        trusters = {a for a, b in trust}
        candidates = [p for p in range(1, n + 1) if p not in trusters]
        if len(candidates) != 1:
            return -1
        c = candidates[0]
        count = sum(1 for a, b in trust if b == c)
        return c if count == n - 1 else -1

print(SolutionEliminate().findJudge(4, [[1, 3], [1, 4], [2, 3], [2, 4], [4, 3]]))
print(SolutionEliminate().findJudge(1, []))

3
1


## 9. Vì sao đúng? (Cách A)

**Bổ đề 1.** Với mọi `p`: `in[p] ≤ n − 1` (chỉ có `n − 1` người khác, cặp không trùng, `a ≠ b`) và `out[p] ≥ 0`. Do đó `score[p] = in[p] − out[p] ≤ n − 1`, với dấu bằng **khi và chỉ khi** `in[p] = n − 1` và `out[p] = 0` — tức `p` thoả điều kiện 1 và 2.

**Bổ đề 2.** Không thể có hai người `p ≠ q` cùng thoả 1 và 2: `in[p] = n − 1` nghĩa là `q` tin `p` → `out[q] ≥ 1`, trái với điều kiện 1 của `q`. Vậy điều kiện 3 tự động thoả khi có người đạt `n − 1`.

**Đoạn lọc:** nếu `len(trust) < n − 1` thì không ai có `in = n − 1` → `-1` đúng.

Do đó trả về `p` đầu tiên có `score = n − 1` là chính xác, và `-1` khi không có.

## 10. Độ phức tạp

Gọi `m = len(trust)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Brute force | O(n·m) | O(1) | dễ nghĩ |
| **A. Điểm số** | **O(n + m)** | O(n) | một mảng, gọn nhất |
| B. Hai mảng bậc | O(n + m) | O(n) | rõ nghĩa đồ thị nhất |
| C. Set + Counter | O(n + m) | O(n) | dùng thư viện |
| D. Loại trừ | O(n + m) | O(n) | tư duy "celebrity" |

## 11. Bộ test

In [13]:
tests = [
    (2, [[1, 2]], 2),
    (3, [[1, 3], [2, 3]], 3),
    (3, [[1, 3], [2, 3], [3, 1]], -1),
    (1, [], 1),                                       # một người, không ai khác
    (2, [], -1),                                      # không ai tin ai
    (3, [[1, 2], [2, 3]], -1),                        # dây chuyền
    (4, [[1, 3], [1, 4], [2, 3], [2, 4], [4, 3]], 3),
    (3, [[1, 2], [2, 1]], -1),                        # tin lẫn nhau
    (3, [[1, 3], [2, 1]], -1),                        # 3 không được 2 tin
]
impls = {
    "brute": find_judge_brute, "chính": find_judge,
    "LC-A": Solution().findJudge, "LC-B": SolutionDegrees().findJudge,
    "LC-C": SolutionSet().findJudge, "LC-D": SolutionEliminate().findJudge,
}
for n, trust, expected in tests:
    for name, f in impls.items():
        got = f(n, trust)
        assert got == expected, (name, n, trust, got, expected)
    print(f"n={n}, trust={str(trust):40} -> {expected} PASS")
print("Tất cả test PASS!")

n=2, trust=[[1, 2]]                                 -> 2 PASS
n=3, trust=[[1, 3], [2, 3]]                         -> 3 PASS
n=3, trust=[[1, 3], [2, 3], [3, 1]]                 -> -1 PASS
n=1, trust=[]                                       -> 1 PASS
n=2, trust=[]                                       -> -1 PASS
n=3, trust=[[1, 2], [2, 3]]                         -> -1 PASS
n=4, trust=[[1, 3], [1, 4], [2, 3], [2, 4], [4, 3]] -> 3 PASS
n=3, trust=[[1, 2], [2, 1]]                         -> -1 PASS
n=3, trust=[[1, 3], [2, 1]]                         -> -1 PASS
Tất cả test PASS!


In [14]:
import random
rng = random.Random(997)
for _ in range(2000):
    n = rng.randint(1, 7)
    pairs = [[a, b] for a in range(1, n + 1) for b in range(1, n + 1) if a != b]
    if rng.random() < 0.5 and n > 1:              # cố ý cài một thẩm phán (có thể bị phá sau)
        j = rng.randint(1, n)
        must = [[a, j] for a in range(1, n + 1) if a != j]
        rest = [p for p in pairs if p[0] != j and p not in must]
        trust = must + rng.sample(rest, rng.randint(0, len(rest)))
        if rng.random() < 0.3:
            trust.append([j, rng.choice([p for p in range(1, n + 1) if p != j])])
    else:
        trust = rng.sample(pairs, rng.randint(0, len(pairs)))
    rng.shuffle(trust)
    expected = find_judge_brute(n, trust)
    for name, f in impls.items():
        assert f(n, trust) == expected, (name, n, trust)
print("2000 test ngẫu nhiên: mọi cách khớp brute force")

2000 test ngẫu nhiên: mọi cách khớp brute force


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra bậc vào

In [15]:
def wrong_in_only(n, trust):
    in_deg = [0] * (n + 1)
    for a, b in trust:
        in_deg[b] += 1
    for p in range(1, n + 1):
        if in_deg[p] == n - 1:
            return p
    return -1

print("Sai :", wrong_in_only(3, [[1, 3], [2, 3], [3, 1]]))
print("Đúng:", find_judge(3, [[1, 3], [2, 3], [3, 1]]))

Sai : 3
Đúng: -1


**Vì sao sai:** 3 được mọi người tin nhưng 3 lại tin 1 → vi phạm điều kiện 1. **Sửa:** kiểm tra cả `out = 0`, hoặc dùng `score = in − out`.

### Lỗi 2: Mảng dài `n` và dùng nhãn trực tiếp làm chỉ số

In [16]:
def wrong_size(n, trust):
    score = [0] * n                 # thiếu 1 ô
    for a, b in trust:
        score[a] -= 1
        score[b] += 1
    return score

try:
    wrong_size(2, [[1, 2]])
except IndexError as e:
    print("Lỗi:", e)

Lỗi: list index out of range


**Sửa:** `[0] * (n + 1)` và duyệt `range(1, n + 1)`; hoặc dùng `score[a - 1]` nhất quán.

### Lỗi 3: Quên trường hợp `n = 1`, `trust = []`

In [17]:
def wrong_empty(n, trust):
    if not trust:
        return -1                    # sai khi n = 1
    return find_judge(n, trust)

print("Sai :", wrong_empty(1, []))
print("Đúng:", find_judge(1, []))

Sai : -1
Đúng: 1


**Vì sao sai:** với `n = 1`, người duy nhất không tin ai và `n − 1 = 0` người khác cần tin → là thẩm phán. Không cần xử lý riêng — công thức `score == n − 1 == 0` tự đúng.

### Lỗi 4: Duyệt `range(n)` → bỏ sót người `n` và xét người 0 không tồn tại

Với `n = 2`, `trust = [[1,2]]`, `range(2)` chỉ xét 0 và 1 → không bao giờ tìm thấy người 2.

## 13. Mẹo nhận dạng pattern

- Dữ liệu dạng **cặp quan hệ** `[a, b]` ("a tin b", "a theo dõi b", "a thắng b") → nghĩ đến **đồ thị có hướng**.
- Điều kiện kiểu "**mọi người** trỏ tới X, X **không** trỏ tới ai" → **bậc vào / bậc ra**.
- Hai điều kiện đếm cùng chiều có thể gộp thành **một hiệu số** khi biết chặn trên/dưới.
- Nhãn `1..n` → mảng `n + 1` ô, bỏ ô 0.

## 14. Biến thể

1. **LeetCode 277 — Find the Celebrity (Premium):** chỉ được hỏi `knows(a, b)`; dùng loại trừ hai con trỏ O(n) lần hỏi.
2. **LeetCode 1557 — Minimum Number of Vertices to Reach All Nodes:** các đỉnh có bậc vào 0.
3. **LeetCode 1615 — Maximal Network Rank:** dùng bậc của đỉnh.
4. **LeetCode 2374 — Node With Highest Edge Score.**

In [18]:
def find_celebrity(n, knows):
    # Biến thể 1: knows(a, b) -> bool. Người nổi tiếng: ai cũng biết, không biết ai.
    cand = 0
    for i in range(1, n):
        if knows(cand, i):        # cand biết i -> cand không thể là celebrity
            cand = i
    for i in range(n):
        if i != cand and (knows(cand, i) or not knows(i, cand)):
            return -1
    return cand

graph = {(0, 2), (1, 2), (3, 2), (0, 1)}
print("Celebrity:", find_celebrity(4, lambda a, b: (a, b) in graph))

Celebrity: 2


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Bậc vào
Trả về list `in_deg` với `in_deg[p]` = số người tin `p` (ô 0 = 0).

In [19]:
def in_degrees(n: int, trust: list[list[int]]) -> list[int]:
    # TODO
    pass

# assert in_degrees(3, [[1, 3], [2, 3]]) == [0, 0, 0, 2]

### Bài 2 — Những người không tin ai
Trả về list (tăng dần) những người không tin ai.

In [20]:
def trust_nobody(n: int, trust: list[list[int]]) -> list[int]:
    # TODO
    pass

# assert trust_nobody(4, [[1, 3], [2, 3]]) == [3, 4]

### Bài 3 — Người được tin nhiều nhất
Trả về nhãn người có bậc vào lớn nhất (hoà thì nhãn nhỏ nhất).

In [21]:
def most_trusted(n: int, trust: list[list[int]]) -> int:
    # TODO
    pass

# assert most_trusted(4, [[1, 2], [3, 2], [1, 4], [2, 4]]) == 2

### Bài 4 — Tin lẫn nhau
Đếm số cặp `{a, b}` mà `a` tin `b` **và** `b` tin `a`.

In [22]:
def mutual_pairs(trust: list[list[int]]) -> int:
    # TODO
    pass

# assert mutual_pairs([[1, 2], [2, 1], [1, 3], [3, 2], [2, 3]]) == 2

### Bài 5 — Thẩm phán "lỏng"
Trả về mọi người `p` không tin ai và được **ít nhất** `k` người tin (list tăng dần).

In [23]:
def loose_judges(n: int, trust: list[list[int]], k: int) -> list[int]:
    # TODO
    pass

# assert loose_judges(4, [[1, 3], [2, 3], [1, 4]], 1) == [3, 4]
# assert loose_judges(4, [[1, 3], [2, 3], [1, 4]], 2) == [3]

### Bài 6 (nâng cao) — Find the Celebrity
Viết lại `celebrity(n, knows)` (người 0..n-1) gọi `knows` tối đa `3n` lần, không nhìn mục 14.

In [24]:
def celebrity(n: int, knows) -> int:
    # TODO
    pass

# g = {(0, 2), (1, 2), (3, 2), (0, 1)}
# assert celebrity(4, lambda a, b: (a, b) in g) == 2
# assert celebrity(2, lambda a, b: True) == -1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | mảng `n + 1` ô | duyệt `trust` | `in_deg[b] += 1` |
| 2 | ai xuất hiện ở vị trí `a`? | set `trusters` | lọc `range(1, n+1)` không thuộc set |
| 3 | bậc vào | `max` với `key` | `max(range(1, n+1), key=lambda p: in_deg[p])` (max trả phần tử đầu tiên khi hoà) |
| 4 | tra nhanh cặp ngược | set các tuple `(a, b)` | đếm `(b, a) in s` với `a < b` |
| 5 | như bài chính | điều kiện `out == 0 and in >= k` | gom thành list |
| 6 | loại trừ: nếu `cand` biết `i` thì `cand` hết hy vọng | 1 lượt chọn ứng viên | 1 lượt kiểm tra lại ứng viên |

## 17. Đáp án tham khảo

In [25]:
def ans_in_degrees(n, trust):
    d = [0] * (n + 1)
    for a, b in trust:
        d[b] += 1
    return d

def ans_trust_nobody(n, trust):
    trusters = {a for a, b in trust}
    return [p for p in range(1, n + 1) if p not in trusters]

def ans_most_trusted(n, trust):
    d = ans_in_degrees(n, trust)
    return max(range(1, n + 1), key=lambda p: d[p])

def ans_mutual_pairs(trust):
    s = {(a, b) for a, b in trust}
    return sum(1 for a, b in s if a < b and (b, a) in s)

def ans_loose_judges(n, trust, k):
    ind, outd = [0] * (n + 1), [0] * (n + 1)
    for a, b in trust:
        outd[a] += 1
        ind[b] += 1
    return [p for p in range(1, n + 1) if outd[p] == 0 and ind[p] >= k]

def ans_celebrity(n, knows):
    cand = 0
    for i in range(1, n):
        if knows(cand, i):
            cand = i
    for i in range(n):
        if i != cand and (knows(cand, i) or not knows(i, cand)):
            return -1
    return cand

assert ans_in_degrees(3, [[1, 3], [2, 3]]) == [0, 0, 0, 2]
assert ans_trust_nobody(4, [[1, 3], [2, 3]]) == [3, 4]
assert ans_most_trusted(4, [[1, 2], [3, 2], [1, 4], [2, 4]]) == 2
assert ans_mutual_pairs([[1, 2], [2, 1], [1, 3], [3, 2], [2, 3]]) == 2
assert ans_loose_judges(4, [[1, 3], [2, 3], [1, 4]], 1) == [3, 4]
assert ans_loose_judges(4, [[1, 3], [2, 3], [1, 4]], 2) == [3]
g = {(0, 2), (1, 2), (3, 2), (0, 1)}
assert ans_celebrity(4, lambda a, b: (a, b) in g) == 2
assert ans_celebrity(2, lambda a, b: True) == -1
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findJudge(self, n: int, trust: list[list[int]]) -> int:
        score = [0] * (n + 1)
        for a, b in trust:
            score[a] -= 1          # a tin người khác -> mất tư cách
            score[b] += 1          # b được tin
        for p in range(1, n + 1):
            if score[p] == n - 1:
                return p
        return -1
```

Câu thần chú: **"Được tin cộng một, tin người trừ một; thẩm phán đạt n − 1."**

**Checklist trước khi submit**
- [ ] Kiểm tra cả "không tin ai" (out = 0), không chỉ bậc vào?
- [ ] Mảng dài `n + 1`, duyệt `range(1, n + 1)`?
- [ ] `n = 1, trust = []` → trả `1`?
- [ ] `n = 2, trust = []` → trả `-1`?
- [ ] Không có ai đạt → `-1`?